# Config A: Custom CNN trained from scratch

Baseline for the comparison in Part f. A small 4-block CNN with random initial weights,
trained only on the Dogs vs. Cats training images, then scored on the val images.

**How to run:** Kernel → Restart, then Run All.

- `FULL_RUN = False`: 2,000 train / 500 val images, for testing on your own laptop.
- `FULL_RUN = True`: all 20,000 / 5,000 images. **Only the trainer sets this.**

Outputs (paths fixed by `common.py`):

- `outputs/results_A_seed42.csv` (one row per epoch)
- `outputs/preds_A_seed42.csv` (per-image val predictions)
- `checkpoints/customcnn_A_seed42.pth` (best val accuracy; goes to the shared Drive, not GitHub)

In [ ]:
FULL_RUN = False   # trainer only: set True for the final run

In [ ]:
import pandas as pd
import torch
import torch.nn as nn

from common import *

set_seed()
print("device:", DEVICE)

## 1. Model

| Stage | Layers | Output shape |
|---|---|---|
| Input | RGB image, 224×224 | (B, 3, 224, 224) |
| Block 1 | Conv 3×3 (32) → BatchNorm → ReLU → MaxPool 2×2 | (B, 32, 112, 112) |
| Block 2 | Conv 3×3 (64) → BatchNorm → ReLU → MaxPool 2×2 | (B, 64, 56, 56) |
| Block 3 | Conv 3×3 (128) → BatchNorm → ReLU → MaxPool 2×2 | (B, 128, 28, 28) |
| Block 4 | Conv 3×3 (256) → BatchNorm → ReLU → MaxPool 2×2 | (B, 256, 14, 14) |
| Pool | AdaptiveAvgPool 4×4 | (B, 256, 4, 4) |
| Classifier | Flatten → Linear 4096→512 → ReLU → Dropout 0.5 → Linear 512→2 | (B, 2) logits |

Notes for the report:

- Conv layers have no bias because the BatchNorm right after them has its own shift, so a bias would do nothing.
- Conv weights use Kaiming (He) normal initialisation, which is designed for ReLU networks and keeps
  activations from shrinking or exploding at the start of training from scratch.
- Output is 2 logits (cat = 0, dog = 1), trained with `CrossEntropyLoss`.

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class CustomCNN(nn.Module):
    """Config A: 4 conv blocks + 2 fully connected layers, trained from scratch."""

    def __init__(self, num_classes: int = 2, dropout: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(self.features(x))

### Shape check

Passes a dummy batch through the model and prints the output shape of every stage.
This table and the parameter count go into the architecture figure for Part b.

In [ ]:
def layer_shapes(model, input_shape=(2, 3, 224, 224)):
    """Output shape and parameter count of each stage, from one dummy forward pass."""
    stages = [(f"features.{n}", m) for n, m in model.features.named_children()]
    stages += [(f"classifier.{n}", m) for n, m in model.classifier.named_children()]
    rows, hooks = [], []
    for name, module in stages:
        def hook(mod, inp, out, name=name):
            rows.append({"stage": name, "layer": mod.__class__.__name__,
                         "output_shape": tuple(out.shape),
                         "params": sum(p.numel() for p in mod.parameters())})
        hooks.append(module.register_forward_hook(hook))
    model.eval()
    with torch.no_grad():
        out = model(torch.zeros(input_shape))
    for h in hooks:
        h.remove()
    return out, pd.DataFrame(rows)


out, shapes = layer_shapes(CustomCNN())
print(shapes.to_string(index=False))
assert out.shape == (2, 2), f"expected (2, 2), got {tuple(out.shape)}"
print(f"\noutput shape {tuple(out.shape)} OK | total params {shapes.params.sum():,}")

## 2. Training settings

| Setting | Value | Reason |
|---|---|---|
| Optimizer | Adam, lr = 1e-3 | Random starting weights need larger steps than pretrained ones (Config C uses 1e-4) |
| Scheduler | StepLR, ×0.5 every 3 epochs | Same schedule as every Dogs vs. Cats config; halves the LR 4 times in 15 epochs |
| Max epochs | 15 | Same training budget as Config C, so A vs C is a fair comparison |
| Checkpoint kept | Best val accuracy | "Up to 15 epochs", same rule for every config |
| Train augmentation | Yes | Same shared transforms as Config C |
| Loss | CrossEntropyLoss | Standard loss for class logits |
| Batch size | 32 | Fixed for every config in `common.py` |
| BatchNorm | All layers in train mode | Nothing is pretrained, so every BN layer learns its statistics |

LR per epoch: 1e-3 (epochs 1–3) → 5e-4 (4–6) → 2.5e-4 (7–9) → 1.25e-4 (10–12) → 6.25e-5 (13–15).

**Report note (A vs C):** both the architecture and the pretraining differ, so the gap shows their
combined effect, not pretraining alone.

In [ ]:
CONFIG = "A"
MODEL_NAME = "customcnn"   # checkpoint: checkpoints/customcnn_A_seed42.pth
EPOCHS = 15
LR = 1e-3
STEP_SIZE = 3
GAMMA = 0.5

In [ ]:
train_loader, val_loader = get_dataloaders(FULL_RUN)

In [ ]:
set_seed()   # same initial weights on every run
model = CustomCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)
criterion = nn.CrossEntropyLoss()

history = fit(model, CONFIG, train_loader, val_loader, optimizer, criterion,
              epochs=EPOCHS, scheduler=scheduler,
              keep_bn_train=None,       # scratch model: all BatchNorm layers learn
              model_name=MODEL_NAME)
history[["epoch", "train_loss", "train_acc", "eval_loss", "eval_acc", "lr", "seconds"]]

## 3. Val predictions

`fit()` has already loaded the best epoch back into `model`. This saves a prediction for every
val image, for the confusion matrix and error analysis in `03_comparison.ipynb`.

In [ ]:
best = history.loc[history.eval_acc.idxmax()]
print(f"best val accuracy {best.eval_acc:.4f} at epoch {int(best.epoch)} of {EPOCHS}")
preds = predict_val(model, val_loader, CONFIG)